Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from google import genai
from google.genai import types
import base64

client = userdata.get('GOOGLE_API_KEY')

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=["Say hello."]
)
print(response.text)

Hello!


Install independencies

In [3]:
!pip install google-genai --break-system-packages

In [ ]:
from google import genai

client = userdata.get('GOOGLE_API_KEY')

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=["Say hello."]
)

print(response.text)

Hello! How can I help you today?


Load the model

For one image

In [ ]:
from google import genai
from google.genai import types
import base64
import re

client = userdata.get('GOOGLE_API_KEY')

image_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images/31.jpg"

with open(image_path, "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Part.from_bytes(data=base64.b64decode(image_data), mime_type="image/jpeg"),
        prompt_text
    ]
)

raw = response.text.strip()
print("RAW OUTPUT:", raw)

m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
label = m.group(1) if m else None
print("PARSED LABEL:", label)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

For multi images

删除错误结果

In [ ]:
import json

with open("/content/drive/MyDrive/Gemini25Flash_HM_ZeroShot_pred.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

print(f"当前已保存: {len(predictions)} 条")

当前已保存: 17 条


In [ ]:
import os
import json
import base64
import re
import time
from google import genai
from google.genai import types
from tqdm import tqdm

client = userdata.get('GOOGLE_API_KEY')

image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
output_json = "/content/drive/MyDrive/Gemini25Flash_HM_ZeroShot_pred.json"

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

def call_with_retry(image_data, mime, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[
                    types.Part.from_bytes(data=base64.b64decode(image_data), mime_type=mime),
                    prompt_text
                ]
            )
            return response.text.strip()
        except Exception as e:
            if "503" in str(e) or "429" in str(e):
                wait = 10 * (attempt + 1)
                print(f"  服务器忙，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

image_files = sorted(
    [f for f in os.listdir(image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(image_dir, img_name)

    try:
        ext = img_name.lower().split(".")[-1]
        mime = "image/png" if ext == "png" else "image/gif" if ext == "gif" else "image/jpeg"

        with open(img_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")

        raw = call_with_retry(image_data, mime)

        m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
        if m:
            label = m.group(1)
        elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
            label = "Non_LGBT"
        else:
            label = "UNKNOWN"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(1)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(3)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

发现已有成功结果 16 条，从断点继续...
剩余待处理: 216 张


推理进度:   0%|          | 0/216 [00:00<?, ?it/s]

  服务器忙，等待 10 秒后重试...
  服务器忙，等待 20 秒后重试...
  服务器忙，等待 30 秒后重试...
  服务器忙，等待 40 秒后重试...
  服务器忙，等待 50 秒后重试...
❌ 17.jpg 出错: 超过最大重试次数


推理进度:   0%|          | 1/216 [02:46<9:55:21, 166.15s/it]

  服务器忙，等待 10 秒后重试...
  服务器忙，等待 20 秒后重试...
  服务器忙，等待 30 秒后重试...
  服务器忙，等待 40 秒后重试...
  服务器忙，等待 50 秒后重试...
❌ 18.jpeg 出错: 超过最大重试次数


...

✅ 238.jpg -> Non_LGBT


推理进度: 100%|█████████▉| 215/216 [2:31:58<00:16, 16.25s/it]

  服务器忙，等待 10 秒后重试...
✅ 239.jpg -> Non_LGBT


推理进度: 100%|██████████| 216/216 [2:32:19<00:00, 42.31s/it]


完成！共 232 条结果已保存
  ERROR: 24
  Homophobia: 64
  Non_LGBT: 134
  Transphobia: 10


In [ ]:
import os
import json
import base64
import re
import time
from google import genai
from google.genai import types

client = userdata.get('GOOGLE_API_KEY')

image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
output_json = "/content/drive/MyDrive/Gemini25Flash_HM_ZeroShot_pred.json"

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

error_items = [p for p in predictions if p["predicted_label"] == "ERROR"]
print(f"需要重跑: {len(error_items)} 张")

for p in error_items:
    img_name = p["image_name"]
    img_path = os.path.join(image_dir, img_name)
    print(f"重跑: {img_name}")

    for attempt in range(5):
        try:
            ext = img_name.lower().split(".")[-1]
            mime = "image/png" if ext == "png" else "image/gif" if ext == "gif" else "image/jpeg"

            with open(img_path, "rb") as f:
                image_data = base64.b64encode(f.read()).decode("utf-8")

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[
                    types.Part.from_bytes(data=base64.b64decode(image_data), mime_type=mime),
                    prompt_text
                ]
            )
            raw = response.text.strip()

            m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
            if m:
                label = m.group(1)
            elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
                label = "Non_LGBT"
            else:
                label = "UNKNOWN"

            p["predicted_label"] = label
            p["raw_output"] = raw
            print(f"✅ {img_name} -> {label}")
            time.sleep(1)
            break

        except Exception as e:
            print(f"  第{attempt+1}次失败: {str(e)[:100]}")
            time.sleep(30)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

labels = [p["predicted_label"] for p in predictions]
print(f"\n最终统计:")
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

需要重跑: 24 张
重跑: 17.jpg
✅ 17.jpg -> Non_LGBT
重跑: 18.jpeg
✅ 18.jpeg -> Non_LGBT
重跑: 20.jpg
✅ 20.jpg -> Homophobia
重跑: 24.jpg
  第1次失败: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman
✅ 24.jpg -> Homophobia
重跑: 25.jpg
✅ 25.jpg -> Homophobia
重跑: 27.jpg
✅ 27.jpg -> Non_LGBT
重跑: 28.jpg
✅ 28.jpg -> Non_LGBT
重跑: 29.jpg
✅ 29.jpg -> Non_LGBT
重跑: 30.jpg
✅ 30.jpg -> Non_LGBT
重跑: 31.jpg
✅ 31.jpg -> Non_LGBT
重跑: 34.jpg
✅ 34.jpg -> Homophobia
重跑: 35.jpg
✅ 35.jpg -> Non_LGBT
重跑: 36.jpeg
✅ 36.jpeg -> Homophobia
重跑: 37.jpg
✅ 37.jpg -> Non_LGBT
重跑: 41.jpg
  第1次失败: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman
✅ 41.jpg -> Non_LGBT
重跑: 45.jpeg
✅ 45.jpeg -> Non_LGBT
重跑: 46.jpeg
✅ 46.jpeg -> Homophobia
重跑: 47.jpg
✅ 47.jpg -> Non_LGBT
重跑: 48.jpeg
✅ 48.jpeg -> Non_LGBT
重跑: 49.jpg
✅ 49.jpg -> Non_LGBT
重跑: 52.jpg
✅ 52.jpg -> Transphobia
重跑: 55.jpg
✅ 55.jpg -> Non_LGBT
重跑: 59.jpeg
✅ 59.jpeg -> Non_LGBT
重跑: 11

In [ ]:
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ===============================
# 1️⃣ 读取预测结果
# ===============================
json_path = "/content/drive/MyDrive/Gemini25Flash_HM_ZeroShot_pred.json"

with open(json_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

pred_df = pd.DataFrame(predictions)
pred_df["image_id"] = (
    pred_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .str.replace(".jpeg", "", regex=False)
    .str.replace(".png", "", regex=False)
    .str.replace(".gif", "", regex=False)
    .str.strip()
)

# ===============================
# 2️⃣ 读取真实标签
# ===============================
excel_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_labels.xlsx"
true_df = pd.read_excel(excel_path)
true_df.columns = true_df.columns.str.strip().str.lower()
true_df["image_id"] = true_df["id"].astype(str).str.strip()

# ===============================
# 3️⃣ 合并
# ===============================
merged_df = pd.merge(true_df, pred_df, on="image_id", how="inner")

print("Total True Labels:", len(true_df))
print("Total Predictions:", len(pred_df))
print("After Merge:", len(merged_df))

# ===============================
# 4️⃣ 标签标准化
# ===============================
def normalize_label(x):
    x = str(x).strip().lower()
    if x in ["homophobia", "homophobic"]:
        return "homophobic"
    if x in ["transphobia", "transphobic"]:
        return "transphobic"
    if x in ["non_lgbt", "non-above", "non_anti_lgbt", "non anti lgbt"]:
        return "non anti lgbt"
    return x

merged_df["true_label"] = merged_df["label"].apply(normalize_label)
merged_df["predicted_label"] = merged_df["predicted_label"].apply(normalize_label)

print("\nUnique TRUE labels:", sorted(merged_df["true_label"].unique()))
print("Unique PRED labels:", sorted(merged_df["predicted_label"].unique()))

# ===============================
# 5️⃣ 计算指标
# ===============================
y_true = merged_df["true_label"]
y_pred = merged_df["predicted_label"]

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Total True Labels: 239
Total Predictions: 232
After Merge: 232

Unique TRUE labels: ['homophobic', 'non anti lgbt', 'transphobic']
Unique PRED labels: ['homophobic', 'non anti lgbt', 'transphobic']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.5517
MP   : 0.7142
MR   : 0.6857
MF1  : 0.5986
WP   : 0.8464
WR   : 0.5517
WF1  : 0.5736
-------------------------------------------------

Confusion Matrix:
[[70 97  2]
 [ 0 49  0]
 [ 0  5  9]]

Classification Report:
               precision    recall  f1-score   support

   homophobic       1.00      0.41      0.59       169
non anti lgbt       0.32      1.00      0.49        49
  transphobic       0.82      0.64      0.72        14

     accuracy                           0.55       232
    macro avg       0.71      0.69      0.60       232
 weighted avg       0.85      0.55      0.57       232



Few-shot

In [ ]:
import os
import json
import base64
import re
import time
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
from google import genai
from google.genai import types

client = userdata.get('GOOGLE_API_KEY')

# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
train_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Train/Train_images"
output_json = "/content/drive/MyDrive/Gemini25Flash_HM_FewShot_RAG_pred.json"

# ===============================
# 加载 CLIP + 训练集 embeddings
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

train_embeddings = np.load("/content/drive/MyDrive/train_embeddings.npy")
with open("/content/drive/MyDrive/train_meta.json", "r") as f:
    train_meta = json.load(f)

train_labels = train_meta["labels"]
train_filenames = train_meta["filenames"]
print(f"训练集 embeddings 加载完成，共 {len(train_embeddings)} 条")

# ===============================
# RAG 检索函数
# ===============================
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        emb = outputs.pooler_output
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().numpy()

def retrieve_examples(test_emb):
    similarities = train_embeddings @ test_emb
    examples = {}
    for label in ["Homophobic", "Transphobic", "Non_Anti_LGBT"]:
        label_indices = [i for i, l in enumerate(train_labels) if l == label]
        label_sims = [(i, similarities[i]) for i in label_indices]
        label_sims.sort(key=lambda x: x[1], reverse=True)
        examples[label] = label_sims[0][0]
    return examples

def build_prompt(example_indices):
    label_map = {
        "Homophobic": "Homophobia",
        "Transphobic": "Transphobia",
        "Non_Anti_LGBT": "Non_LGBT"
    }
    examples_text = ""
    for i, (label, idx) in enumerate(example_indices.items()):
        examples_text += f"Example {i+1}: Class label: {label_map[label]}\n"

    return f"""You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Below are 3 reference examples with their correct labels retrieved from similar memes:

{examples_text}
Now classify the following meme:

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text, using the provided examples as reference to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

# ===============================
# API 调用（带重试）
# ===============================
def call_with_retry(image_data, mime, prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[
                    types.Part.from_bytes(data=base64.b64decode(image_data), mime_type=mime),
                    prompt
                ]
            )
            return response.text.strip()
        except Exception as e:
            if "503" in str(e) or "429" in str(e):
                wait = 10 * (attempt + 1)
                print(f"  服务器忙，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

# ===============================
# 获取测试图片列表
# ===============================
image_files = sorted(
    [f for f in os.listdir(test_image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

# ===============================
# 批量推理
# ===============================
for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(test_image_dir, img_name)

    try:
        # RAG 检索
        test_emb = get_embedding(img_path)
        example_indices = retrieve_examples(test_emb)
        prompt_text = build_prompt(example_indices)

        ext = img_name.lower().split(".")[-1]
        mime = "image/png" if ext == "png" else "image/gif" if ext == "gif" else "image/jpeg"

        with open(img_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")

        raw = call_with_retry(image_data, mime, prompt_text)

        m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
        if m:
            label = m.group(1)
        elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
            label = "Non_LGBT"
        else:
            label = "UNKNOWN"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(1)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(3)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

Loading CLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

✅ 238.jpg -> Non_LGBT


推理进度: 100%|█████████▉| 231/232 [27:34<00:09,  9.06s/it]

✅ 239.jpg -> Non_LGBT


推理进度: 100%|██████████| 232/232 [27:39<00:00,  7.15s/it]


完成！共 232 条结果已保存
  Homophobia: 71
  Non_LGBT: 151
  Transphobia: 10


 Calculation of Few shot

In [ ]:
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

json_path = "/content/drive/MyDrive/Gemini25Flash_HM_FewShot_RAG_pred.json"

with open(json_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

pred_df = pd.DataFrame(predictions)
pred_df["image_id"] = (
    pred_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .str.replace(".jpeg", "", regex=False)
    .str.replace(".png", "", regex=False)
    .str.replace(".gif", "", regex=False)
    .str.strip()
)

excel_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_labels.xlsx"
true_df = pd.read_excel(excel_path)
true_df.columns = true_df.columns.str.strip().str.lower()
true_df["image_id"] = true_df["id"].astype(str).str.strip()

merged_df = pd.merge(true_df, pred_df, on="image_id", how="inner")

print("Total True Labels:", len(true_df))
print("Total Predictions:", len(pred_df))
print("After Merge:", len(merged_df))

def normalize_label(x):
    x = str(x).strip().lower()
    if x in ["homophobia", "homophobic"]:
        return "homophobic"
    if x in ["transphobia", "transphobic"]:
        return "transphobic"
    if x in ["non_lgbt", "non-above", "non_anti_lgbt", "non anti lgbt"]:
        return "non anti lgbt"
    return x

merged_df["true_label"] = merged_df["label"].apply(normalize_label)
merged_df["predicted_label"] = merged_df["predicted_label"].apply(normalize_label)

print("\nUnique TRUE labels:", sorted(merged_df["true_label"].unique()))
print("Unique PRED labels:", sorted(merged_df["predicted_label"].unique()))

y_true = merged_df["true_label"]
y_pred = merged_df["predicted_label"]

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Total True Labels: 239
Total Predictions: 232
After Merge: 232

Unique TRUE labels: ['homophobic', 'non anti lgbt', 'transphobic']
Unique PRED labels: ['homophobic', 'non anti lgbt', 'transphobic']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.5560
MP   : 0.7701
MR   : 0.7095
MF1  : 0.6356
WP   : 0.8471
WR   : 0.5560
WF1  : 0.5787
-------------------------------------------------

Confusion Matrix:
[[70 99  0]
 [ 0 49  0]
 [ 1  3 10]]

Classification Report:
               precision    recall  f1-score   support

   homophobic       0.99      0.41      0.58       169
non anti lgbt       0.32      1.00      0.49        49
  transphobic       1.00      0.71      0.83        14

     accuracy                           0.56       232
    macro avg       0.77      0.71      0.64       232
 weighted avg       0.85      0.56      0.58       232



V1-V2-V3跑对齐

In [4]:
import json, re
from sklearn.metrics import confusion_matrix, classification_report

base = '/content/drive/MyDrive/'
ct   = '/content/drive/MyDrive/MyThesis2026/Chinese/Test/'
LAB  = ['homophobic', 'transphobic', 'non anti lgbt']

def norm(x):
    x = x.strip().lower().replace('_',' ')
    return {'homophobia':'homophobic','transphobia':'transphobic',
            'non lgbt':'non anti lgbt','non anti lgbt':'non anti lgbt'}.get(x, x)

v1 = json.load(open(base + 'Gemini25Flash_HM_ZeroShot_pred.json'))
v3 = json.load(open(ct + 'step2_v3_full_results.json'))
id2gt = {r['image_id']: norm(r['ground_truth']) for r in v3}
def nid(n):
    m = re.search(r'(\d+)', n); return int(m.group(1)) if m else None

v1c = [(id2gt[nid(p['image_name'])], norm(p['predicted_label']))
       for p in v1 if p['predicted_label'] not in ('ERROR','UNKNOWN') and nid(p['image_name']) in id2gt]
yt1, yp1 = zip(*v1c)
print("=== V1 [Homo, Trans, Non] ===  n =", len(v1c))
print(confusion_matrix(yt1, yp1, labels=LAB))
print(classification_report(yt1, yp1, labels=LAB, digits=4, zero_division=0))

v3c = [(norm(r['ground_truth']), norm(r['final_label'])) for r in v3]
yt3, yp3 = zip(*v3c)
print("=== V3 [Homo, Trans, Non] ===  n =", len(v3c))
print(confusion_matrix(yt3, yp3, labels=LAB))
print(classification_report(yt3, yp3, labels=LAB, digits=4, zero_division=0))

=== V1 [Homo, Trans, Non] ===  n = 232
[[70  2 97]
 [ 0  9  5]
 [ 0  0 49]]
               precision    recall  f1-score   support

   homophobic     1.0000    0.4142    0.5858       169
  transphobic     0.8182    0.6429    0.7200        14
non anti lgbt     0.3245    1.0000    0.4900        49

     accuracy                         0.5517       232
    macro avg     0.7142    0.6857    0.5986       232
 weighted avg     0.8464    0.5517    0.5736       232

=== V3 [Homo, Trans, Non] ===  n = 239
[[114   3  59]
 [  0  12   2]
 [  0   0  49]]
               precision    recall  f1-score   support

   homophobic     1.0000    0.6477    0.7862       176
  transphobic     0.8000    0.8571    0.8276        14
non anti lgbt     0.4455    1.0000    0.6164        49

     accuracy                         0.7322       239
    macro avg     0.7485    0.8350    0.7434       239
 weighted avg     0.8746    0.7322    0.7538       239

